# SELE Score Model Training

Files are read from and written to your Google Drive folder.

**Before running:**
1. Place these files in the same Drive folder:
   - `model_definition.py` — from `src/regularization/score_model/`
   - `sele_simulated_100000_curves_500_long.mat` (d500) or `sele_simulated_100000_curves_32_long.mat` (d32) — from `Data/score_model/`
2. Set `DRIVE_DIR` in the config cell to your folder path.

Outputs (checkpoints + final `.pt`) are saved to the same Drive folder — persistent across sessions.

In [1]:
# Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')

CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── CONFIGURATION ────────────────────────────────────────────
MODEL_SIZE = 'd500'   # <── change to 'd32' or 'd500'
CHECKPOINT_EVERY = 10  # Save resume checkpoint every N epochs
DRIVE_DIR = '/content/drive/MyDrive/Thesis/Colab Model Training'  # <── change to your folder
# ─────────────────────────────────────────────────────────────

_DATA_FILE = {
    'd32':  'sele_simulated_100000_curves_500_long.mat',
    'd500': 'sele_simulated_100000_curves_500_long.mat',
}
_OUTPUT_FILE = {
    'd32':  'sele_score_net_d32.pt',
    'd500': 'sele_score_net_d500.pt',
}
_RESUME_FILE = f'checkpoint_resume_{MODEL_SIZE}.pt'

import os
data_path   = os.path.join(DRIVE_DIR, _DATA_FILE[MODEL_SIZE])
output_path = os.path.join(DRIVE_DIR, _OUTPUT_FILE[MODEL_SIZE])
resume_path = os.path.join(DRIVE_DIR, _RESUME_FILE)

assert os.path.exists(data_path), (
    f'Data file not found: {data_path}\n'
    f'Place it in your Drive folder: {DRIVE_DIR}'
)
assert os.path.exists(os.path.join(DRIVE_DIR, 'model_definition.py')), (
    f'model_definition.py not found in {DRIVE_DIR}\n'
    'Upload it from src/regularization/score_model/model_definition.py'
)
print(f'Config OK — training {MODEL_SIZE} model')
print(f'  data   : {data_path}')
print(f'  output : {output_path}')

Config OK — training d500 model
  data   : /content/drive/MyDrive/Thesis/Colab Model Training/sele_simulated_100000_curves_500_long.mat
  output : /content/drive/MyDrive/Thesis/Colab Model Training/sele_score_net_d500.pt


In [4]:
import sys
import os
import logging
import time
from dataclasses import dataclass, asdict
from typing import Tuple

import numpy as np
import scipy.io
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

sys.path.insert(0, DRIVE_DIR)
from model_definition import ScoreNetwork  # loaded from Drive folder

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print('Imports OK')

Imports OK


In [5]:
_PRESETS = {
    'd32': dict(
        target_length=32,
        hidden_dims=(64, 128, 256, 256, 128, 64),
        use_layer_norm=True,
        use_residual=False,  # Legacy architecture for backward compat with existing d32 checkpoints
        use_time_embedding=False,
        time_embed_dim=128,
        batch_size=256,
        learning_rate=1e-3,
        num_epochs=100,
    ),
    'd500': dict(
        target_length=500,
        hidden_dims=(512, 1024, 2048, 2048, 1024, 512),
        use_layer_norm=True,
        use_residual=True,  # Residual connections — critical for d500 convergence
        use_time_embedding=True,  # Sinusoidal time embedding for richer time conditioning
        time_embed_dim=128,
        batch_size=512,  # Reduced from 1024 — more gradient noise helps escape flat loss
        learning_rate=1e-3,
        num_epochs=300,  # Increased from 100 — d500 needs more training time
    ),
}


@dataclass
class TrainingConfig:
    target_length: int
    hidden_dims: Tuple[int, ...]
    use_layer_norm: bool
    use_residual: bool
    use_time_embedding: bool
    time_embed_dim: int
    batch_size: int
    learning_rate: float
    num_epochs: int
    data_path: str
    output_path: str
    beta_min: float = 0.1
    beta_max: float = 20.0
    time_eps: float = 1e-4


class DiffusionModel:
    def __init__(self, score_network, config, device=None):
        self.score_network = score_network
        self.config = config
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.score_network.to(self.device)
        print(f'Training on device: {self.device}')

    def compute_diffusion_params(self, t, x):
        int_beta = (self.config.beta_min + 0.5 * (self.config.beta_max - self.config.beta_min) * t) * t
        mu_t = x * torch.exp(-0.5 * int_beta)
        var_t = -torch.expm1(-int_beta)
        return mu_t, var_t

    def compute_loss(self, x):
        t = torch.rand((x.shape[0], 1), dtype=x.dtype, device=x.device) * (1 - self.config.time_eps) + self.config.time_eps
        mu_t, var_t = self.compute_diffusion_params(t, x)
        x_t = mu_t + torch.sqrt(var_t) * torch.randn_like(x)
        grad_log_p = -(x_t - mu_t) / var_t
        score = self.score_network(x_t, t)
        return (var_t * (score - grad_log_p) ** 2).mean()

    def train_epoch(self, data_loader, optimizer):
        self.score_network.train()
        total_loss, n = 0.0, 0
        for batch, in data_loader:
            batch = batch.to(self.device)
            optimizer.zero_grad()
            loss = self.compute_loss(batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.score_network.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * batch.shape[0]
            n += batch.shape[0]
        return total_loss / n


print('Classes defined')

Classes defined


In [6]:
torch.manual_seed(42)
np.random.seed(42)

# Load data
print(f'Loading {data_path}')
mat = scipy.io.loadmat(data_path)
data = torch.tensor(mat['data'], dtype=torch.float32)
print(f'Data shape: {data.shape}')

data_min, data_max = data.min().item(), data.max().item()
print(f'min={data_min:.4e}  max={data_max:.4e}')
data = 2 * (data - data_min) / (data_max - data_min) - 1

preset = _PRESETS[MODEL_SIZE]
target_length = preset['target_length']
original_len = data.shape[1]
if original_len != target_length:
    indices = torch.linspace(0, original_len - 1, target_length).round().long()
    data = data[:, indices]
    print(f'Downsampled {original_len} → {target_length}')

config = TrainingConfig(**preset, data_path=data_path, output_path=output_path)
data_loader = DataLoader(TensorDataset(data), batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)

score_network = ScoreNetwork(
    input_dim=config.target_length + 1,
    output_dim=config.target_length,
    hidden_dims=config.hidden_dims,
    use_layer_norm=config.use_layer_norm,
    use_residual=config.use_residual,
    use_time_embedding=config.use_time_embedding,
    time_embed_dim=config.time_embed_dim,
)

# Compile for faster GPU execution (PyTorch 2.0+)
try:
    score_network = torch.compile(score_network)
    print('torch.compile() enabled')
except Exception:
    print('torch.compile() not available, skipping')

diffusion_model = DiffusionModel(score_network, config)
optimizer = torch.optim.Adam(score_network.parameters(), lr=config.learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs, eta_min=1e-5)

start_epoch = 0
if os.path.exists(resume_path):
    print(f'Resuming from {resume_path}')
    ckpt = torch.load(resume_path, map_location=diffusion_model.device)
    score_network.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resuming from epoch {start_epoch}/{config.num_epochs}')
else:
    print('Starting fresh')

n_params = sum(p.numel() for p in score_network.parameters())
print(f'Ready — {MODEL_SIZE}, {config.num_epochs} epochs, batch={config.batch_size}, lr={config.learning_rate}')
print(f'Architecture: residual={config.use_residual}, time_embed={config.use_time_embedding}, params={n_params:,}')

Loading /content/drive/MyDrive/Thesis/Colab Model Training/sele_simulated_100000_curves_500_long.mat
Data shape: torch.Size([100000, 500])
min=2.7800e-21  max=3.4765e-02
torch.compile() enabled
Training on device: cuda
Starting fresh
Ready — d500, 300 epochs, batch=512, lr=0.001
Architecture: residual=True, time_embed=True, params=15,606,260


In [7]:
# ── TRAINING LOOP ─────────────────────────────────────────────
start_time = time.time()

for epoch in tqdm(range(start_epoch, config.num_epochs), desc=f'Training {MODEL_SIZE}', initial=start_epoch, total=config.num_epochs):
    epoch_loss = diffusion_model.train_epoch(data_loader, optimizer)
    scheduler.step()

    print(
        f'Epoch {epoch+1}/{config.num_epochs} | '
        f'Loss: {epoch_loss:.6f} | '
        f'LR: {scheduler.get_last_lr()[0]:.2e} | '
        f'Elapsed: {time.time()-start_time:.0f}s'
    )

    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': score_network.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': asdict(config),
            'data_min': data_min,
            'data_max': data_max,
        }, resume_path)
        print(f'Resume checkpoint saved (epoch {epoch+1})')

print(f'Training done in {time.time()-start_time:.0f}s')

Training d500:   0%|          | 0/300 [00:00<?, ?it/s]

W0320 13:32:20.355000 3257 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1/300 | Loss: 6.040182 | LR: 1.00e-03 | Elapsed: 30s
Epoch 2/300 | Loss: 0.179169 | LR: 1.00e-03 | Elapsed: 34s
Epoch 3/300 | Loss: 0.132470 | LR: 1.00e-03 | Elapsed: 38s
Epoch 4/300 | Loss: 0.114456 | LR: 1.00e-03 | Elapsed: 42s
Epoch 5/300 | Loss: 0.103522 | LR: 9.99e-04 | Elapsed: 46s
Epoch 6/300 | Loss: 0.092967 | LR: 9.99e-04 | Elapsed: 49s
Epoch 7/300 | Loss: 0.086326 | LR: 9.99e-04 | Elapsed: 53s
Epoch 8/300 | Loss: 0.080247 | LR: 9.98e-04 | Elapsed: 57s
Epoch 9/300 | Loss: 0.075118 | LR: 9.98e-04 | Elapsed: 61s
Epoch 10/300 | Loss: 0.071137 | LR: 9.97e-04 | Elapsed: 65s
Resume checkpoint saved (epoch 10)
Epoch 11/300 | Loss: 0.069162 | LR: 9.97e-04 | Elapsed: 70s
Epoch 12/300 | Loss: 0.066967 | LR: 9.96e-04 | Elapsed: 74s
Epoch 13/300 | Loss: 0.061458 | LR: 9.95e-04 | Elapsed: 78s
Epoch 14/300 | Loss: 0.430788 | LR: 9.95e-04 | Elapsed: 82s
Epoch 15/300 | Loss: 0.067696 | LR: 9.94e-04 | Elapsed: 86s
Epoch 16/300 | Loss: 0.058047 | LR: 9.93e-04 | Elapsed: 90s
Epoch 17/300 |

In [8]:
# ── SAVE FINAL MODEL ──────────────────────────────────────────
torch.save({
    'model_state_dict': score_network.state_dict(),
    'config': asdict(config),
    'data_min': data_min,
    'data_max': data_max,
}, output_path)
print(f'Saved → {output_path}')

if os.path.exists(resume_path):
    os.remove(resume_path)
    print('Resume checkpoint deleted')

print(f'\nDone! Download {output_path} from the file panel')
print(f'Then place it in Data/score_model/ in your local repo.')

Saved → /content/drive/MyDrive/Thesis/Colab Model Training/sele_score_net_d500.pt
Resume checkpoint deleted

Done! Download /content/drive/MyDrive/Thesis/Colab Model Training/sele_score_net_d500.pt from the file panel
Then place it in Data/score_model/ in your local repo.
